# Business Queries

## Overview
Analytical queries on the gold layer dimensional model for business insights and operational monitoring.

**Purpose**: Demonstrate Galaxy schema analytics for device performance, API metrics, asset maintenance, and regional analysis.

---

## Device Health Overview
Aggregate device performance metrics by device and region.

In [0]:
SELECT
    d.device_id,
    d.device_type,
    d.region,
    COUNT(*) AS total_events,
    ROUND(AVG(f.cpu_usage), 2) AS avg_cpu_usage,
    ROUND(AVG(f.memory_usage), 2) AS avg_memory_usage,
    ROUND(AVG(f.packet_loss), 2) AS avg_packet_loss,
    ROUND(AVG(f.latency_ms), 2) AS avg_latency,
    MAX(f.temp) AS max_temperature

FROM telecom_catalog.gold_schema.fact_device_logs f

JOIN telecom_catalog.gold_schema.dim_device d
        ON f.device_key = d.device_key

GROUP BY d.device_id,
        d.device_type,
        d.region

ORDER BY avg_cpu_usage DESC;

---
## Regional Device Performance
Aggregate performance metrics grouped by region to identify regional trends.

In [0]:
SELECT
    d.region,
    COUNT(*) AS total_events,
    ROUND(AVG(f.cpu_usage), 2) AS avg_cpu_usage,
    ROUND(AVG(f.memory_usage), 2) AS avg_memory_usage,
    ROUND(AVG(f.packet_loss), 2) AS avg_packet_loss,
    ROUND(AVG(f.latency_ms), 2) AS avg_latency
    
FROM telecom_catalog.gold_schema.fact_device_logs f

JOIN telecom_catalog.gold_schema.dim_device d
    ON f.device_key = d.device_key

GROUP BY d.region
ORDER BY avg_cpu_usage DESC;

---
## Network Health Analysis
Count of network health status by region to assess overall network quality.

In [0]:
%sql
SELECT
    d.region,
    f.network_health,
    COUNT(*) AS event_count

FROM telecom_catalog.gold_schema.fact_device_logs f

JOIN telecom_catalog.gold_schema.dim_device d
    ON f.device_key = d.device_key

GROUP BY
    d.region,
    f.network_health

ORDER BY
    d.region,
    event_count DESC;

---
## Asset Maintenance Analysis
Breakdown of asset maintenance status and requirements by region.

In [0]:
%sql
SELECT
    a.region,
    a.maintenance_status,
    a.maintenance_required,

    COUNT(*) AS asset_count

FROM telecom_catalog.gold_schema.dim_asset a

GROUP BY
    a.region,
    a.maintenance_status,
    a.maintenance_required

ORDER BY
    a.region;

---
## Critical Assets
Identify assets requiring maintenance, prioritized by SLA level.

In [0]:
%sql
SELECT
    a.asset_key,
    d.device_id,
    d.device_type,

    a.asset_type,
    a.vendor,
    a.firmware_version,
    a.maintenance_status,
    a.maintenance_required,
    a.sla_level,
    a.sla_priority,
    a.region

FROM telecom_catalog.gold_schema.dim_asset a

JOIN telecom_catalog.gold_schema.dim_device d
    ON a.device_key = d.device_key

WHERE
    a.maintenance_required = 'Yes'

ORDER BY
    a.sla_priority ASC;

---
## API Performance by Date
Daily API performance trends including latency, packet loss, and request counts.

In [0]:
%sql
SELECT
    t.full_date,

    COUNT(*) AS total_events,

    ROUND(AVG(f.latency_ms_p99), 2) AS avg_latency_p99,
    ROUND(AVG(f.packet_loss_percentage), 2) AS avg_packet_loss,
    ROUND(AVG(f.cache_hit_ratio), 2) AS avg_cache_hit_ratio,

    SUM(f.failed_requests) AS total_failed_requests,
    SUM(f.request_count) AS total_requests

FROM telecom_catalog.gold_schema.fact_api_metrics f

JOIN telecom_catalog.gold_schema.dim_time t
    ON f.time_key = t.time_key

GROUP BY t.full_date

ORDER BY t.full_date;

---
## API Failure Rate
Calculate daily API failure rate as percentage of failed requests.

In [0]:
%sql
SELECT
    t.full_date,

    SUM(f.failed_requests) AS failed_requests,
    SUM(f.request_count) AS total_requests,

    ROUND(
        SUM(f.failed_requests) * 100.0
        / NULLIF(SUM(f.request_count), 0),
        2
    ) AS failure_rate_percentage

FROM telecom_catalog.gold_schema.fact_api_metrics f

JOIN telecom_catalog.gold_schema.dim_time t
    ON f.time_key = t.time_key

GROUP BY t.full_date

ORDER BY t.full_date;

---
## Top Problematic Devices
Identify the top 10 devices with highest failure rates and latency.

In [0]:
%sql
SELECT
    d.device_id,
    d.device_type,
    d.region,

    COUNT(*) AS total_events,

    SUM(f.failed_requests) AS failed_requests,

    ROUND(AVG(f.latency_ms_p99), 2) AS avg_latency_p99,

    ROUND(AVG(f.packet_loss_percentage), 2) AS avg_packet_loss

FROM telecom_catalog.gold_schema.fact_api_metrics f

JOIN telecom_catalog.gold_schema.dim_device d
    ON f.device_key = d.device_key

GROUP BY
    d.device_id,
    d.device_type,
    d.region

ORDER BY
    failed_requests DESC

LIMIT 10;

---
## Summary

This notebook demonstrates analytical queries on the gold layer dimensional model.

**Query Categories**:
* Device performance monitoring (CPU, memory, latency, packet loss)
* API performance analysis (request counts, failure rates, latency trends)
* Network health assessment by region
* Asset maintenance tracking and critical asset identification
* Regional operational analysis

**Use Cases**: BI dashboards, operational monitoring, data-driven infrastructure management.